# 🚀 PEFT 라이브러리 연동 심화 실습

In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

import logging
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

import transformers
transformers.logging.set_verbosity_error()

import torch
import torch.nn as nn
import math

## 1. 현업 표준: HuggingFace PEFT 로드 및 적용
앞선 노트북에서 직접 밑바닥부터 짠 LoRA 로직을, 실제 산업 현장에서는 `peft`라는 라이브러리 단 몇 줄의 코드로 적용합니다. 소형 모델인 `distilgpt2`에 적용해 보겠습니다.

In [2]:
from transformers import AutoModelForCausalLM
from peft import get_peft_model, LoraConfig, TaskType

# 1. 원본 모델 로드
model_name = "distilgpt2"
print(f"{model_name} 원본 모델 로딩 중...")
model = AutoModelForCausalLM.from_pretrained(model_name)

# 2. LoRA 설정 (Config)
# r=8, lora_alpha=32가 가장 대중적인 세팅입니다.
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["c_attn"], # GPT-2 구조에서 Attention 모듈의 가중치 이름
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM # 인과적 언어 생성 모델
)

# 3. 모델을 PEFT 모델로 변환
print("모델에 LoRA 어댑터 장착 중...")
peft_model = get_peft_model(model, lora_config)

# 4. 파라미터 효율성 확인
peft_model.print_trainable_parameters()
print("\n👉 보시다시피 수백만 개의 파라미터가 전부 동결되고, 단 몇십만 개만 학습 가능(Trainable) 상태로 변경되었습니다. 이제 이 가벼운 모델을 가지고 일반적인 GPU에서도 거대 언어 모델을 파인튜닝할 수 있습니다!")

distilgpt2 원본 모델 로딩 중...


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

모델에 LoRA 어댑터 장착 중...
trainable params: 147,456 || all params: 82,060,032 || trainable%: 0.1797

👉 보시다시피 수백만 개의 파라미터가 전부 동결되고, 단 몇십만 개만 학습 가능(Trainable) 상태로 변경되었습니다. 이제 이 가벼운 모델을 가지고 일반적인 GPU에서도 거대 언어 모델을 파인튜닝할 수 있습니다!
